# ELM Validation: Benchmark Results

17 LLMs (16 open-weight + 1 proprietary reference) evaluated on 41 ELM-CPG pairs: 16 valid, 13 parametric invalid, 12 semantic invalid (4 categories: missing condition, inverted logic, wrong nesting, swapped references).

All results are means over 5 independent trials at temperature 0.1. Base rate (naive always-valid): 39.0%.

In [1]:
import pandas as pd, numpy as np, os, json, warnings
from scipy import stats
warnings.filterwarnings('ignore')

MT_DIR = None
for d in ['../results/multi_trial', 'results/multi_trial']:
    if os.path.isdir(d): MT_DIR = d; break

GT_PATH = None
for p in ['../test_data/ground_truth.json', 'test_data/ground_truth.json']:
    if os.path.exists(p): GT_PATH = p; break

# Categorize cases: valid / parametric / semantic
with open(GT_PATH) as f:
    gt = json.load(f)
categories = {}
for fname, tc in gt['test_cases'].items():
    if tc['valid']: categories[fname] = 'valid'
    elif 'semantic' in tc.get('notes','').lower(): categories[fname] = 'semantic'
    else: categories[fname] = 'parametric'

# Model tiers (ordered by accuracy within each tier)
FRONTIER = ['gemma-4-26b-a4b', 'gemma-4-31b', 'qwen3-32b', 'gpt-oss-120b',
            'qwen3.5-35b-a3b', 'llama-3.3-70b', 'gpt-oss-20b']
PROPRIETARY = ['gpt-5.4-mini']
MIDRANGE = ['medgemma-1.5-4b', 'phi-3-mini', 'gemma-3-4b', 'medgemma-4b']
SMALL = ['llama-3.2-1b', 'qwen-2.5-3b', 'llama-3.2-3b', 'qwen-2.5-1.5b']
ALL_MODELS = FRONTIER + PROPRIETARY + MIDRANGE + SMALL

# Load all multi-trial data (skip few-shot variants)
frames = []
for f in sorted(os.listdir(MT_DIR)):
    if not (f.startswith('results-') and f.endswith('.csv')): continue
    if 'few-shot' in f: continue
    df = pd.read_csv(os.path.join(MT_DIR, f))
    for col in ['valid', 'correct', 'expected_valid']:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().str.lower() == 'true'
    # Skip all-error trial files
    if (df['source'] == 'error').all(): continue
    frames.append(df)
data = pd.concat(frames, ignore_index=True)

n_cases = data['file'].nunique()
n_valid = sum(1 for c in categories.values() if c == 'valid')
n_param = sum(1 for c in categories.values() if c == 'parametric')
n_semantic = sum(1 for c in categories.values() if c == 'semantic')

print(f"Multi-trial: {data['model'].nunique()} models x {data['trial'].nunique()} trials x {n_cases} cases")
print(f"Cases: {n_valid} valid, {n_param} parametric invalid, {n_semantic} semantic invalid")
print(f"Base rate (always-valid): {n_valid/n_cases:.1%}")

Multi-trial: 16 models x 5 trials x 41 cases
Cases: 16 valid, 13 parametric invalid, 12 semantic invalid
Base rate (always-valid): 39.0%


## Per-Model Accuracy (Mean ± SD over 5 trials) with Per-Category Disaggregation

In [2]:
def compute_metrics(mdf):
    """Compute per-trial metrics and return DataFrame of trials."""
    per_trial = []
    for t in sorted(mdf['trial'].unique()):
        tdf = mdf[mdf['trial'] == t]
        cv = cp = cs = tv_cnt = tp_cnt = ts = 0
        tp = fp = tn = fn = 0
        for _, r in tdf.iterrows():
            cat = categories.get(r['file'], 'unknown')
            c = bool(r['correct'])
            ev = bool(r['expected_valid'])
            pv = bool(r['valid'])
            if ev and pv: tp += 1
            elif ev and not pv: fn += 1
            elif not ev and pv: fp += 1
            else: tn += 1
            if cat == 'valid': tv_cnt += 1; cv += c
            elif cat == 'parametric': tp_cnt += 1; cp += c
            elif cat == 'semantic': ts += 1; cs += c
        per_trial.append({
            'acc': tdf['correct'].mean(),
            'sens': tp/(tp+fn) if (tp+fn) else 0,
            'spec': tn/(tn+fp) if (tn+fp) else 0,
            'f1': 2*tp/(2*tp+fp+fn) if (2*tp+fp+fn) else 0,
            'valid_acc': cv/tv_cnt if tv_cnt else 0,
            'param_acc': cp/tp_cnt if tp_cnt else 0,
            'semantic_acc': cs/ts if ts else 0,
        })
    return pd.DataFrame(per_trial)

rows = []
for model in ALL_MODELS:
    mdf = data[data['model'] == model]
    if len(mdf) == 0: continue
    ptdf = compute_metrics(mdf)
    rows.append({
        'Model': model,
        'Accuracy': f"{ptdf['acc'].mean():.1%}",
        '±SD': f"{ptdf['acc'].std():.1%}",
        'Sens': f"{ptdf['sens'].mean():.2f}",
        'Spec': f"{ptdf['spec'].mean():.2f}",
        'F1': f"{ptdf['f1'].mean():.2f}",
        'Valid': f"{ptdf['valid_acc'].mean():.0%}",
        'Param': f"{ptdf['param_acc'].mean():.0%}",
        'Semantic': f"{ptdf['semantic_acc'].mean():.0%}",
    })

pd.DataFrame(rows)

,Model,Accuracy,±SD,Sens,Spec,F1,Valid,Param,Semantic
0,gemma-4-26b-a4b,80.5%,0.0%,0.62,0.92,0.71,62%,100%,83%
1,gemma-4-31b,82.9%,0.0%,0.69,0.92,0.76,69%,85%,100%
2,qwen3-32b,90.7%,2.0%,0.88,0.93,0.88,88%,100%,85%
3,gpt-oss-120b,85.9%,2.0%,0.64,1.00,0.78,64%,100%,100%
4,qwen3.5-35b-a3b,85.4%,3.4%,0.69,0.96,0.78,69%,100%,92%
5,llama-3.3-70b,83.4%,1.1%,0.88,0.81,0.80,88%,100%,60%
6,gpt-oss-20b,79.5%,4.1%,0.60,0.92,0.70,60%,92%,92%
7,gpt-5.4-mini,81.0%,3.2%,0.51,1.00,0.67,51%,100%,100%
8,medgemma-1.5-4b,70.2%,6.1%,0.64,0.74,0.62,64%,83%,65%
9,phi-3-mini,50.7%,2.7%,0.99,0.20,0.61,99%,37%,2%


## Fisher's Exact Test: Frontier vs Small

In [3]:
# Fisher's exact test using majority vote across 5 trials
mv = data.groupby(['model', 'file']).agg(
    n_correct=('correct', 'sum'),
    expected_valid=('expected_valid', 'first')
).reset_index()
mv['correct'] = mv['n_correct'] >= 3

frontier_mv = mv[mv['model'].isin(FRONTIER)]
midrange_mv = mv[mv['model'].isin(MIDRANGE)]
small_mv = mv[mv['model'].isin(SMALL)]

lc = int(frontier_mv['correct'].sum()); lt = len(frontier_mv)
mc = int(midrange_mv['correct'].sum()); mt_ = len(midrange_mv)
sc = int(small_mv['correct'].sum()); st = len(small_mv)

print(f"Open-weight Frontier (7 models, majority vote over 5 trials): {lc}/{lt} ({lc/lt:.1%})")
print(f"Mid-range (4 models, majority vote): {mc}/{mt_} ({mc/mt_:.1%})")
print(f"Small (4 models, majority vote):     {sc}/{st} ({sc/st:.1%})")
print()

odds1, p1 = stats.fisher_exact([[lc, lt - lc], [mc, mt_ - mc]])
print(f"Frontier vs Mid-range: OR={odds1:.2f}, p={p1:.2e}")
odds2, p2 = stats.fisher_exact([[lc, lt - lc], [sc, st - sc]])
print(f"Frontier vs Small:     OR={odds2:.2f}, p={p2:.2e}")

Open-weight Frontier (7 models, majority vote over 5 trials): 243/287 (84.7%)
Mid-range (4 models, majority vote): 87/164 (53.0%)
Small (4 models, majority vote):     62/164 (37.8%)

Frontier vs Mid-range: OR=4.89, p=9.90e-13
Frontier vs Small:     OR=9.09, p=3.62e-24


## McNemar Pairwise Tests (Bonferroni corrected)

In [4]:
from itertools import combinations

# McNemar pairwise tests using majority vote across 5 trials
functioning = [m for m in ALL_MODELS if m in mv['model'].unique()]
pairs = list(combinations(functioning, 2))
alpha_corr = 0.05 / len(pairs)
print(f"{len(pairs)} pairs, Bonferroni alpha={alpha_corr:.5f}\n")

sig_pairs = []
for m1, m2 in pairs:
    d1 = mv[mv['model'] == m1].set_index('file')['correct']
    d2 = mv[mv['model'] == m2].set_index('file')['correct']
    common = d1.index.intersection(d2.index)
    b = int((d1[common] & ~d2[common]).sum())
    c = int((~d1[common] & d2[common]).sum())
    nd = b + c
    p = 1.0 if nd == 0 else stats.binomtest(max(b, c), nd, 0.5).pvalue
    if p < 0.05:
        sig_pairs.append({
            'Pair': f'{m1} vs {m2}', 'b': b, 'c': c,
            'p': f'{p:.4f}', 'Sig': '**' if p < alpha_corr else '*'
        })

if sig_pairs:
    print(f"Pairs reaching uncorrected significance (p<0.05): {len(sig_pairs)}")
    print("  ** = Bonferroni-corrected significant, * = uncorrected only")
    display(pd.DataFrame(sig_pairs))
else:
    print("No pairs reached p<0.05")

120 pairs, Bonferroni alpha=0.00042



Pairs reaching uncorrected significance (p<0.05): 65
  ** = Bonferroni-corrected significant, * = uncorrected only


,Pair,b,c,p,Sig
0,gemma-4-26b-a4b vs phi-3-mini,18,6,0.0227,*
1,gemma-4-26b-a4b vs gemma-3-4b,19,6,0.0146,*
2,gemma-4-26b-a4b vs medgemma-4b,23,6,0.0023,*
3,gemma-4-26b-a4b vs llama-3.2-1b,21,5,0.0025,*
4,gemma-4-26b-a4b vs qwen-2.5-3b,23,6,0.0023,*
...,...,...,...,...,...
60,medgemma-1.5-4b vs llama-3.2-3b,20,5,0.0041,*
61,medgemma-1.5-4b vs qwen-2.5-1.5b,20,4,0.0015,*
62,phi-3-mini vs llama-3.2-3b,6,0,0.0312,*
63,phi-3-mini vs qwen-2.5-1.5b,7,0,0.0156,*
